# NeRF Evaluation

This notebook evaluates the synthetic **Lego** experiments and compares the **original Poster NeRF (P0)** with the **combined improved Poster NeRF (P2)**.

The evaluation uses four image quality metrics:

- **MSE** — Mean Squared Error
- **PSNR** — Peak Signal-to-Noise Ratio
- **SSIM** — Structural Similarity Index
- **LPIPS** — Learned Perceptual Image Patch Similarity


## Setup

Device setup and initialization of the pretrained AlexNet LPIPS model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lpips

from skimage.metrics import (
    mean_squared_error,
    peak_signal_noise_ratio,
    structural_similarity,
)

In [ ]:
ROOT = Path.cwd()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Repository:", ROOT)
print("Device:", device)

# LPIPS using pretrained AlexNet
lpips_model = lpips.LPIPS(
    net="alex"
).to(device)

lpips_model.eval()

print("LPIPS loaded successfully.")

## Metric Implementation

The evaluation pipeline loads the saved NeRF predictions and ground-truth images, and calculates MSE, PSNR, SSIM and LPIPS for each test view.

- https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.peak_signal_noise_ratio, MSE, PSNR, SSIM
- https://github.com/richzhang/PerceptualSimilarity/blob/master/test_network.py, LPIPS

In [ ]:
def load_test_images(path):
    """
    Load saved NeRF predictions and corresponding ground-truth images.

    Expected NPZ structure:
        predictions: [N, H, W, 3]
        targets:     [N, H, W, 3]

    Expected pixel range:
        [0, 1]
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find: {path}"
        )

    data = np.load(path)

    print("NPZ keys:", data.files)

    if (
        "predictions" not in data.files
        or "targets" not in data.files
    ):
        raise KeyError(
            f"Expected 'predictions' and 'targets'. "
            f"Found: {data.files}"
        )

    predictions = data[
        "predictions"
    ].astype(np.float32)

    targets = data[
        "targets"
    ].astype(np.float32)

    if predictions.shape != targets.shape:
        raise ValueError(
            f"Prediction shape {predictions.shape} "
            f"does not match target shape {targets.shape}"
        )

    if (
        predictions.ndim != 4
        or predictions.shape[-1] != 3
    ):
        raise ValueError(
            "Expected image arrays with shape "
            f"[N, H, W, 3], got {predictions.shape}"
        )

    # protect metrics against small numerical values outside the expected image range
    predictions = np.clip(
        predictions,
        0.0,
        1.0
    )

    targets = np.clip(
        targets,
        0.0,
        1.0
    )

    print(f"Loaded: {path}")
    print(f"Number of images: {len(predictions)}")
    print(f"Shape: {predictions.shape}")

    return predictions, targets


def calculate_lpips(prediction, target):
    """
    Calculate LPIPS perceptual distance between
    two RGB images.

    Input:
        prediction: [H, W, 3], range [0, 1]
        target:     [H, W, 3], range [0, 1]

    LPIPS expects:
        [N, C, H, W], range [-1, 1]
    """

    pred_tensor = (
        torch.from_numpy(prediction)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .float()
        .to(device)
    )

    target_tensor = (
        torch.from_numpy(target)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .float()
        .to(device)
    )

    # converts [0, 1] to [-1, 1]
    pred_tensor = (
        pred_tensor * 2.0 - 1.0
    )

    target_tensor = (
        target_tensor * 2.0 - 1.0
    )

    with torch.no_grad():
        score = lpips_model(
            pred_tensor,
            target_tensor
        )

    return score.item()


def evaluate_images(predictions, targets):
    """
    Evaluate all held-out test images.

    Metrics:
        MSE    - lower is better
        PSNR   - higher is better
        SSIM   - higher is better
        LPIPS  - lower is better

    Returns:
        pandas DataFrame with one row per image.
    """

    results = []

    for i, (prediction, target) in enumerate(
        zip(predictions, targets)
    ):

        mse = mean_squared_error(
            target,
            prediction
        )

        psnr = peak_signal_noise_ratio(
            target,
            prediction,
            data_range=1.0
        )

        ssim = structural_similarity(
            target,
            prediction,
            channel_axis=-1,
            data_range=1.0
        )

        lpips_score = calculate_lpips(
            prediction,
            target
        )

        results.append(
            {
                "image": i,
                "MSE": mse,
                "PSNR": psnr,
                "SSIM": ssim,
                "LPIPS": lpips_score,
            }
        )

    return pd.DataFrame(results)


def summarize_metrics(results):
    """
    Return mean and standard deviation
    for all evaluation metrics.
    """

    return results[
        [
            "MSE",
            "PSNR",
            "SSIM",
            "LPIPS",
        ]
    ].agg(
        ["mean", "std"]
    )


print("Evaluation functions loaded.")

## Experiment Setup

Three saved Lego configurations and the original and combined Poster reconstructions are evaluated.


In [ ]:
lego_experiments = {
    "Lego 200px / 10k": (
        ROOT / "lego_output_res200" / "lego_test_images.npz"
    ),
    "Lego 400px / 20k": (
        ROOT / "lego_output_res400" / "lego_test_images.npz"
    ),
    "Lego 400px / 40k / 192 pts": (
        ROOT / "lego_output_res400_40k_pts192" / "lego_test_images.npz"
    ),
}

poster_path = ROOT / "poster_outputs" / "poster_test_images.npz"
poster_combined_path = (
    ROOT
    / "poster_outputs_hierarchical_deep_ipe_dist_0.001"
    / "poster_test_images.npz"
)
poster_baseline_csv = ROOT / "evaluation_results" / "poster_per_view.csv"

print("Available Lego experiments:")
for name, path in lego_experiments.items():
    print(f"{name}: {path.exists()}")

print("Original Poster predictions:", poster_path.exists())
print("Original Poster metrics:", poster_baseline_csv.exists())
print("Combined Poster predictions:", poster_combined_path.exists())


In [ ]:
lego_all_results = {}

lego_summary_rows = []


for experiment_name, experiment_path in lego_experiments.items():

    print()
    print("=" * 70)
    print(experiment_name)
    print("=" * 70)

    predictions, targets = load_test_images(
        experiment_path
    )

    metrics = evaluate_images(
        predictions,
        targets
    )

    lego_all_results[
        experiment_name
    ] = {
        "predictions": predictions,
        "targets": targets,
        "metrics": metrics,
    }

    lego_summary_rows.append(
        {
            "Experiment": experiment_name,
            "MSE": metrics["MSE"].mean(),
            "MSE std": metrics["MSE"].std(),
            "PSNR": metrics["PSNR"].mean(),
            "PSNR std": metrics["PSNR"].std(),
            "SSIM": metrics["SSIM"].mean(),
            "SSIM std": metrics["SSIM"].std(),
            "LPIPS": metrics["LPIPS"].mean(),
            "LPIPS std": metrics["LPIPS"].std(),
        }
    )


lego_summary = pd.DataFrame(
    lego_summary_rows
)

lego_summary

In [ ]:
lego_configuration_table = lego_summary[
    [
        "Experiment",
        "MSE",
        "PSNR",
        "SSIM",
        "LPIPS",
    ]
]

display(
    lego_configuration_table.style.format(
        {
            "MSE": "{:.5f}",
            "PSNR": "{:.2f}",
            "SSIM": "{:.4f}",
            "LPIPS": "{:.4f}",
        }
    )
)

In [ ]:
for _, row in lego_summary.iterrows():

    print(
        row["Experiment"]
    )

    print(
        f"  MSE:   "
        f"{row['MSE']:.5f} ± "
        f"{row['MSE std']:.5f}"
    )

    print(
        f"  PSNR:  "
        f"{row['PSNR']:.2f} ± "
        f"{row['PSNR std']:.2f} dB"
    )

    print(
        f"  SSIM:  "
        f"{row['SSIM']:.4f} ± "
        f"{row['SSIM std']:.4f}"
    )

    print(
        f"  LPIPS: "
        f"{row['LPIPS']:.4f} ± "
        f"{row['LPIPS std']:.4f}"
    )

    print()

## Graphing

PSNR, SSIM and LPIPS are plotted across the Lego configurations.

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    lego_summary["Experiment"],
    lego_summary["PSNR"]
)

plt.ylabel(
    "Mean PSNR (dB)"
)

plt.xlabel(
    "NeRF configuration"
)

plt.title(
    "Lego NeRF Reconstruction Quality"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    lego_summary["Experiment"],
    lego_summary["SSIM"]
)

plt.ylabel(
    "Mean SSIM"
)

plt.xlabel(
    "NeRF configuration"
)

plt.title(
    "Lego NeRF Structural Similarity"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    lego_summary["Experiment"],
    lego_summary["LPIPS"]
)

plt.ylabel(
    "Mean LPIPS"
)

plt.xlabel(
    "NeRF configuration"
)

plt.title(
    "Lego NeRF Perceptual Error"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

plt.show()

## Poster Evaluation

Compare the original Poster reconstruction (P0) with the combined NeRF improvements (P2). The original per-view results are reused when available, avoiding unnecessary reevaluation.


In [ ]:
poster_predictions = None
poster_targets = None

if poster_baseline_csv.exists():
    poster_results = pd.read_csv(poster_baseline_csv)
elif poster_path.exists():
    poster_predictions, poster_targets = load_test_images(poster_path)
    poster_results = evaluate_images(poster_predictions, poster_targets)
else:
    raise FileNotFoundError(
        "Original Poster results not found in evaluation_results/poster_per_view.csv "
        "or poster_outputs/poster_test_images.npz"
    )

poster_combined_predictions, poster_combined_targets = load_test_images(
    poster_combined_path
)
poster_combined_results = evaluate_images(
    poster_combined_predictions, poster_combined_targets
)

if len(poster_results) != len(poster_combined_results):
    raise ValueError("The Poster experiments have different test-view counts.")
if not np.array_equal(
    poster_results["image"].to_numpy(),
    poster_combined_results["image"].to_numpy()
):
    raise ValueError("Poster test-view indices are not aligned.")

print("Original Poster views:", len(poster_results))
print("Combined Poster views:", len(poster_combined_results))
display(poster_combined_results)


In [ ]:
poster_metrics = ["MSE", "PSNR", "SSIM", "LPIPS"]

poster_comparison = pd.DataFrame({
    "Original Poster (P0)": poster_results[poster_metrics].mean(),
    "Combined Poster (P2)": poster_combined_results[poster_metrics].mean(),
}).T
poster_change = (
    poster_comparison.loc["Combined Poster (P2)"]
    - poster_comparison.loc["Original Poster (P0)"]
)

display(poster_comparison.round(5))
print("P2 minus P0:")
display(poster_change.round(5).to_frame(name="Change"))

poster_view_comparison = poster_results[["image"] + poster_metrics].merge(
    poster_combined_results[["image"] + poster_metrics],
    on="image",
    suffixes=("_original", "_combined"),
    validate="one_to_one",
)
poster_view_comparison["PSNR Improvement"] = (
    poster_view_comparison["PSNR_combined"]
    - poster_view_comparison["PSNR_original"]
)

display(poster_view_comparison[
    ["image", "PSNR_original", "PSNR_combined", "PSNR Improvement"]
].round(3))
print("Views with higher PSNR:",
      int((poster_view_comparison["PSNR Improvement"] > 0).sum()),
      "/", len(poster_view_comparison))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
(ROOT / "evaluation_results").mkdir(exist_ok=True)

for ax, metric in zip(axes, ["PSNR", "SSIM", "LPIPS"]):
    values = poster_comparison[metric]
    bars = ax.bar(["Original", "Combined"], values.values)
    ax.set_title(metric)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height(), f"{value:.4f}",
                ha="center", va="bottom")

plt.tight_layout()
plt.savefig(ROOT / "evaluation_results" / "poster_original_vs_combined.png",
            dpi=200, bbox_inches="tight")
plt.show()

## Dataset Comparison

The best recorded Lego configuration is compared with both the original and combined Poster reconstructions.


In [ ]:
BEST_LEGO = (
    "Lego 400px / 40k / 192 pts"
)


lego_predictions = (
    lego_all_results[
        BEST_LEGO
    ]["predictions"]
)

lego_targets = (
    lego_all_results[
        BEST_LEGO
    ]["targets"]
)

lego_results = (
    lego_all_results[
        BEST_LEGO
    ]["metrics"]
)


print(
    "Selected final Lego model:",
    BEST_LEGO
)

In [ ]:
dataset_rows = [{
    "Dataset": "Synthetic Lego",
    "MSE": lego_results["MSE"].mean(),
    "PSNR": lego_results["PSNR"].mean(),
    "SSIM": lego_results["SSIM"].mean(),
    "LPIPS": lego_results["LPIPS"].mean(),
}]

for label, results in [
    ("Real Poster (P0)", poster_results),
    ("Real Poster (P2)", poster_combined_results),
]:
    dataset_rows.append({
        "Dataset": label,
        "MSE": results["MSE"].mean(),
        "PSNR": results["PSNR"].mean(),
        "SSIM": results["SSIM"].mean(),
        "LPIPS": results["LPIPS"].mean(),
    })

dataset_comparison = pd.DataFrame(dataset_rows)
display(dataset_comparison.style.format({
    "MSE": "{:.5f}", "PSNR": "{:.2f}",
    "SSIM": "{:.4f}", "LPIPS": "{:.4f}",
}))


## Qualitative Evaluation

Best and worst test views are compared with ground truth and visualized with error maps.

In [ ]:
lego_best_idx = (
    lego_results["PSNR"].idxmax()
)

lego_worst_idx = (
    lego_results["PSNR"].idxmin()
)


print(
    f"Best Lego view: "
    f"{lego_best_idx}"
)

print(
    f"PSNR: "
    f"{lego_results.loc[lego_best_idx, 'PSNR']:.2f} dB"
)

print()

print(
    f"Worst Lego view: "
    f"{lego_worst_idx}"
)

print(
    f"PSNR: "
    f"{lego_results.loc[lego_worst_idx, 'PSNR']:.2f} dB"
)

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(9, 9)
)


# best ground truth
axes[0, 0].imshow(
    lego_targets[
        lego_best_idx
    ]
)

axes[0, 0].set_title(
    "Best view — Ground truth"
)


# best prediction
axes[0, 1].imshow(
    lego_predictions[
        lego_best_idx
    ]
)

axes[0, 1].set_title(
    f"Prediction — "
    f"PSNR "
    f"{lego_results.loc[lego_best_idx, 'PSNR']:.2f} dB"
)


# worst ground truth
axes[1, 0].imshow(
    lego_targets[
        lego_worst_idx
    ]
)

axes[1, 0].set_title(
    "Worst view — Ground truth"
)


# worst prediction
axes[1, 1].imshow(
    lego_predictions[
        lego_worst_idx
    ]
)

axes[1, 1].set_title(
    f"Prediction — "
    f"PSNR "
    f"{lego_results.loc[lego_worst_idx, 'PSNR']:.2f} dB"
)


for ax in axes.ravel():
    ax.axis("off")


plt.tight_layout()

plt.show()

In [ ]:
idx = lego_worst_idx


prediction = (
    lego_predictions[idx]
)

target = (
    lego_targets[idx]
)


absolute_error = np.abs(
    prediction - target
)


error_map = absolute_error.mean(
    axis=-1
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)


axes[0].imshow(
    target
)

axes[0].set_title(
    "Ground truth"
)


axes[1].imshow(
    prediction
)

axes[1].set_title(
    "NeRF prediction"
)


im = axes[2].imshow(
    error_map
)

axes[2].set_title(
    "Absolute reconstruction error"
)


for ax in axes:
    ax.axis("off")


plt.colorbar(
    im,
    ax=axes[2],
    fraction=0.046,
    pad=0.04
)


plt.tight_layout()

plt.show()

In [ ]:
poster_best_idx = poster_combined_results["PSNR"].idxmax()
poster_worst_idx = poster_combined_results["PSNR"].idxmin()

print("Best combined Poster view:", poster_best_idx)
print(f"PSNR: {poster_combined_results.loc[poster_best_idx, 'PSNR']:.2f} dB")
print("Worst combined Poster view:", poster_worst_idx)
print(f"PSNR: {poster_combined_results.loc[poster_worst_idx, 'PSNR']:.2f} dB")

if poster_predictions is None and poster_path.exists():
    poster_predictions, poster_targets = load_test_images(poster_path)

if poster_targets is not None:
    if poster_targets.shape != poster_combined_targets.shape:
        raise ValueError("Original and combined Poster targets differ in shape.")
    for i in range(len(poster_targets)):
        if not np.allclose(
            poster_targets[i], poster_combined_targets[i],
            rtol=1e-5, atol=1e-5
        ):
            raise ValueError(f"Ground-truth views do not match at index {i}.")


In [ ]:
indices = [poster_best_idx, poster_worst_idx]

fig, axes = plt.subplots(2, 3, figsize=(12, 10))
for row, idx in enumerate(indices):
    axes[row, 0].imshow(poster_combined_targets[idx])
    axes[row, 0].set_title("Ground truth")
    if poster_predictions is not None:
        axes[row, 1].imshow(poster_predictions[idx])
        axes[row, 1].set_title(
            f"Original — {poster_results.loc[idx, 'PSNR']:.2f} dB"
        )
    else:
        axes[row, 1].text(0.5, 0.5, "Original images unavailable",
                          ha="center", va="center")
    axes[row, 2].imshow(poster_combined_predictions[idx])
    axes[row, 2].set_title(
        f"Combined — {poster_combined_results.loc[idx, 'PSNR']:.2f} dB"
    )

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
idx = poster_worst_idx
prediction = poster_combined_predictions[idx]
target = poster_combined_targets[idx]
error_map = np.abs(prediction - target).mean(axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(target)
axes[0].set_title("Ground truth")
axes[1].imshow(prediction)
axes[1].set_title("Combined Poster prediction")
im = axes[2].imshow(error_map)
axes[2].set_title("Absolute reconstruction error")

for ax in axes:
    ax.axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## Export Results

Numerical results and figures are saved for use in the report.

In [ ]:
RESULTS_DIR = ROOT / "evaluation_results"
RESULTS_DIR.mkdir(exist_ok=True)

if "lego_summary" in globals():
    lego_summary.to_csv(
        RESULTS_DIR / "lego_configuration_summary.csv", index=False
    )
if "lego_results" in globals():
    lego_results.to_csv(
        RESULTS_DIR / "lego_best_per_view.csv", index=False
    )
if "dataset_comparison" in globals():
    dataset_comparison.to_csv(
        RESULTS_DIR / "dataset_comparison.csv", index=False
    )

if not poster_baseline_csv.exists():
    poster_results.to_csv(poster_baseline_csv, index=False)
poster_combined_results.to_csv(
    RESULTS_DIR / "poster_combined_per_view.csv", index=False
)
poster_comparison.to_csv(
    RESULTS_DIR / "poster_original_vs_combined.csv"
)
poster_view_comparison.to_csv(
    RESULTS_DIR / "poster_original_vs_combined_per_view.csv", index=False
)

print("Evaluation results saved to:", RESULTS_DIR)


In [ ]:
# PSNR plot
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    lego_summary["Experiment"],
    lego_summary["PSNR"]
)

plt.ylabel(
    "Mean PSNR (dB)"
)

plt.xlabel(
    "NeRF configuration"
)

plt.title(
    "Lego NeRF Reconstruction Quality"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "lego_psnr_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# SSIM plot
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    lego_summary["Experiment"],
    lego_summary["SSIM"]
)

plt.ylabel(
    "Mean SSIM"
)

plt.xlabel(
    "NeRF configuration"
)

plt.title(
    "Lego NeRF Structural Similarity"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "lego_ssim_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# LPIPS plot
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    lego_summary["Experiment"],
    lego_summary["LPIPS"]
)

plt.ylabel(
    "Mean LPIPS"
)

plt.xlabel(
    "NeRF configuration"
)

plt.title(
    "Lego NeRF Perceptual Error"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "lego_lpips_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()